# Veritas ML (`veritas-ml`) — Guia Prático (How to Use)

O **`veritas-ml`** é um framework experimental auditável e **100% agnóstico a modelos e domínio** para comparar representações (fusão fatorial de features) e arquiteturas (individuais ou relacionais/grafos) sob um protocolo estrito sem vazamento de dados.

### Fluxo em 4 Contratos Imutáveis
```text
1. DatasetContext    → Dados, alvos multirrótulo e folds agrupados sem vazamento
2. FeatureContext    → Blocos de features ajustados estritamente no treino de cada fold
3. ExperimentPlan    → Catálogo de combinações de features × modelos registrados no notebook
4. ExperimentResult  → Observações padronizadas por condição/fold/seed → StudyResult & Relatório
```

Este notebook é **100% autocontido** (gera um dataset sintético em memória) e demonstra passo a passo como usar cada módulo do `veritas_ml`.

## 1 — Criando um `DatasetContext` com Folds Agrupados

Você pode carregar seus próprios dados (CSV, Parquet, Banco de Dados) e usar `make_grouped_folds` para garantir que itens do mesmo grupo (ex.: mesma franquia, mesmo cliente, mesmo autor) **nunca** se dividam entre treino e validação.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Garante que src/ esteja no sys.path
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from veritas_ml import (
    DatasetContext,
    FeatureContext,
    GraphContext,
    ModelOutput,
    build_experiment_plan,
    clear_registry,
    fold_summary,
    list_models,
    plan_summary,
    register_model,
)
from veritas_ml.data import make_grouped_folds

# 1. Exemplo de dataset sintético multirrótulo (120 amostras, 24 grupos, 4 classes)
rng = np.random.default_rng(42)
n_samples = 120
classes = ("Classe_A", "Classe_B", "Classe_C", "Classe_D")

temas = [
    ("espaco nave orbita galaxia planeta", [1, 0, 0, 1]),
    ("mercado acoes banco investimento juros", [0, 1, 0, 0]),
    ("esporte torneio equipe quadra campeonato", [0, 0, 1, 0]),
    ("misterio investigacao detetive pista crime", [1, 0, 1, 1]),
]

rows = []
targets_list = []
for i in range(n_samples):
    tema_idx = i % len(temas)
    vocab, base_y = temas[tema_idx]
    palavras = rng.choice(vocab.split(), size=5, replace=True)
    ruido = rng.choice(["contexto", "item", "registro", "dados"], size=2)
    texto = " ".join([*palavras, *ruido])

    rows.append({
        "Id": f"item_{i:03d}",
        "Nome": f"Exemplo {i:03d}",
        "texto": texto,
        "grupo_franquia": f"grupo_{i // 5:02d}",
    })
    # Adiciona leve variação nos rótulos
    y_row = np.logical_xor(base_y, rng.random(len(classes)) < 0.10).astype(np.float32)
    if y_row.sum() == 0:
        y_row[tema_idx] = 1.0
    targets_list.append(y_row)

frame = pd.DataFrame(rows)
targets = np.vstack(targets_list).astype(np.float32)
folds, fold_id = make_grouped_folds(frame["grupo_franquia"], n_splits=5, seed=42)
partitions = pd.DataFrame({"Id": frame["Id"], "particao": "desenvolvimento", "grupo_franquia": frame["grupo_franquia"]})

dataset = DatasetContext(
    frame=frame,
    targets=targets,
    classes=classes,
    folds=tuple(folds),
    fold_id=fold_id,
    partitions=partitions,
    metadata={"dataset_name": "synthetic_demo", "n_development": len(frame), "n_audit_reserved": 0},
)

print(f"Dataset pronto: {len(dataset.frame)} itens | {len(dataset.classes)} classes | {len(dataset.folds)} folds")
display(fold_summary(dataset))

## 2 — Construindo Blocos de Features Fold-Locais (`FeatureContext`)

No `veritas-ml`, cada fonte de representação (ex.: `tfidf_svd`, `dense_emb`, `tabular`) é ajustada **apenas nos índices de treino de cada fold**, evitando vazamento de vocabulário ou normalização. Depois, `combine_blocks` concatena e normaliza (L2) qualquer combinação fatorial de blocos.

In [ ]:
from veritas_ml.features import WORD_CFGS, _fit_svd, _fit_tfidf_base, combine_blocks

# Simulamos uma fonte de embedding denso pré-computado (ex.: vindo de um encoder congelado)
dense_signal = targets @ rng.normal(size=(len(classes), 16)).astype(np.float32)
dense_embeddings = (dense_signal + 1.25 * rng.normal(size=dense_signal.shape)).astype(np.float32)

fold_packs = []
for fold_idx, (tr, va) in enumerate(dataset.folds, start=1):
    texts_tr = dataset.frame.iloc[tr].texto.tolist()
    texts_va = dataset.frame.iloc[va].texto.tolist()

    # Bloco 1: TF-IDF + SVD ajustado apenas no treino do fold
    _, a_tr, a_va, fit_tfidf, trans_tfidf = _fit_tfidf_base(
        texts_tr, texts_va, {"ngram_range": (1, 2), "min_df": 1, "sublinear_tf": True}
    )
    lex_tr, lex_va, _, fit_svd, trans_svd = _fit_svd(a_tr, a_va, target_dim=8, seed=42)

    # Bloco 2: Embedding denso fatiado para o fold
    emb_tr, emb_va = dense_embeddings[tr], dense_embeddings[va]

    fold_packs.append({
        "fold": fold_idx,
        "train": {"lexical": lex_tr, "embedding": emb_tr},
        "valid": {"lexical": lex_va, "embedding": emb_va},
        "meta": {},
        "components": {
            "lexical_base": {"fit_seconds": fit_tfidf + fit_svd, "transform_seconds": trans_tfidf + trans_svd},
            "embedding_lookup": {"fit_seconds": 0.0, "transform_seconds": 0.001},
        },
        "source_components": {
            "lexical": ("lexical_base",),
            "embedding": ("embedding_lookup",),
        },
    })

feature_context = FeatureContext(
    fold_packs=fold_packs,
    fixed_meta={"embedding": {"dim": 16, "source": "synthetic_encoder"}},
)

# Demonstração de fusão fatorial de blocos no Fold 1:
fused_example = combine_blocks(feature_context.fold_packs[0]["train"], ("lexical", "embedding"))
print("Shape do bloco fundido (lexical + embedding):", fused_example.shape)

## 3 — Definindo e Registrando Modelos (`register_model`)

O `veritas-ml` aceita **qualquer modelo** (Scikit-Learn, PyTorch, TensorFlow/Keras, XGBoost) desde que encapsulado em uma função adaptadora com a assinatura:

```python
def adapter(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_eval: np.ndarray,
    graph_context: GraphContext | None,
    seed: int,
) -> ModelOutput
```

- Modelos com `role="individual"` ou `role="individual_baseline"` recebem `graph_context=None`.
- Modelos com `role="relational"` recebem automaticamente `graph_context` contendo as matrizes indutivas `p_train` e `p_eval`.

In [ ]:
import torch
from torch import nn
from sklearn.linear_model import RidgeClassifier

clear_registry()

# --- Modelo 1: Classificador Clássico (Scikit-Learn) ---
def sklearn_ridge_adapter(x_train, y_train, x_eval, graph_context, seed) -> ModelOutput:
    t0 = time.perf_counter()
    clf = RidgeClassifier(random_state=seed)
    clf.fit(x_train, y_train)
    train_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    scores = clf.decision_function(x_eval)
    prob = (1.0 / (1.0 + np.exp(-scores))).astype(np.float32)
    pred_s = time.perf_counter() - t1

    n_params = int(clf.coef_.size + clf.intercept_.size)
    return ModelOutput(prob, n_params, "parameters", train_s, pred_s)


# --- Modelo 2: MLP e GNN em PyTorch (Individual vs Relacional) ---
class SimpleMLPOrGNN(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden: int = 32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    @staticmethod
    def _prop(p: torch.Tensor | None, x: torch.Tensor) -> torch.Tensor:
        return x if p is None else p @ x

    def forward(self, x: torch.Tensor, p: torch.Tensor | None = None) -> torch.Tensor:
        h = torch.relu(self.fc1(self._prop(p, x)))
        return self.fc2(self._prop(p, h))


def pytorch_adapter(x_train, y_train, x_eval, graph_context, seed, use_graph: bool = False) -> ModelOutput:
    torch.manual_seed(seed)
    xt = torch.tensor(x_train, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.float32)
    xe = torch.tensor(np.concatenate([x_train, x_eval]), dtype=torch.float32)

    pt = torch.tensor(graph_context.p_train, dtype=torch.float32) if (use_graph and graph_context) else None
    pe = torch.tensor(graph_context.p_eval, dtype=torch.float32) if (use_graph and graph_context) else None

    net = SimpleMLPOrGNN(xt.shape[1], yt.shape[1], hidden=32)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-2, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    t0 = time.perf_counter()
    for _ in range(60):
        net.train()
        opt.zero_grad()
        loss = loss_fn(net(xt, pt), yt)
        loss.backward()
        opt.step()
    train_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    net.eval()
    with torch.inference_mode():
        logits = net(xe, pe)[len(xt):]
        prob = torch.sigmoid(logits).numpy().astype(np.float32)
    pred_s = time.perf_counter() - t1

    params = sum(p.numel() for p in net.parameters())
    return ModelOutput(prob, params, "parameters", train_s, pred_s)


# Registro dos modelos no catálogo do veritas-ml:
register_model(
    "ridge",
    sklearn_ridge_adapter,
    display_name="Ridge Linear (Sklearn)",
    family="linear",
    role="individual",
    seeds=(42,),
)

register_model(
    "mlp",
    lambda x_tr, y_tr, x_ev, g_ctx, s: pytorch_adapter(x_tr, y_tr, x_ev, g_ctx, s, use_graph=False),
    display_name="PyTorch MLP",
    family="neural",
    role="individual_baseline",
    seeds=(42, 43),
)

register_model(
    "gnn",
    lambda x_tr, y_tr, x_ev, g_ctx, s: pytorch_adapter(x_tr, y_tr, x_ev, g_ctx, s, use_graph=True),
    display_name="Inductive kNN-GNN",
    family="graph",
    role="relational",
    seeds=(42, 43),
)

print("Modelos registrados no Veritas ML:", list_models())

## 4 — Montando o `ExperimentPlan` e Executando a Matriz Fatorial

Definimos quais combinações de features (`registry`) queremos testar contra todos os modelos registrados e executamos a avaliação cruzada.

In [ ]:
from veritas_ml.core.graph import build_knn_propagation
from veritas_ml.engine import standardize_experiment_result
from veritas_ml.features import bundle_component_time
from veritas_ml.models import class_metrics, multilabel_metrics, run_model

# Catálogo fatorial customizado: 2 fontes isoladas + 1 fusão multimodal
custom_registry = [
    {"family": "unimodal", "bundle": ("lexical",), "features": "lexical"},
    {"family": "unimodal", "bundle": ("embedding",), "features": "embedding"},
    {"family": "multimodal", "bundle": ("lexical", "embedding"), "features": "lexical+embedding"},
]

plan = build_experiment_plan(
    dataset,
    persist_registry=False,
    registry=custom_registry,
)
print("Resumo do ExperimentPlan:", plan_summary(plan, dataset))

# Execução em memória de todas as condições (bundles × modelos × folds × seeds)
raw_rows = []
class_rows = []

for entry in plan.registry:
    bundle = entry["bundle"]
    representation = entry["features"]
    family = entry["family"]

    for model_id in plan.model_ids:
        t_cond = time.perf_counter()
        for fold, ((tr, va), pack) in enumerate(zip(dataset.folds, feature_context.fold_packs), start=1):
            t_fuse = time.perf_counter()
            xtr = combine_blocks(pack["train"], bundle)
            xva = combine_blocks(pack["valid"], bundle)
            fusion_s = time.perf_counter() - t_fuse
            fit_s, trans_s, comps = bundle_component_time(pack, bundle)

            # Se o modelo for relacional, constrói o grafo kNN indutivo sem vazamento
            graph_ctx = None
            graph_s = 0.0
            if plan.model_metadata[model_id]["role"] == "relational":
                tg = time.perf_counter()
                p_tr, p_ev, edges_df = build_knn_propagation(xtr, xva, **plan.graph_config)
                graph_s = time.perf_counter() - tg
                graph_ctx = GraphContext(p_train=p_tr, p_eval=p_ev, edges_df=edges_df)

            for seed in plan.model_seeds[model_id]:
                out = run_model(model_id, xtr, dataset.targets[tr], xva, graph_ctx, seed)
                scores = multilabel_metrics(dataset.targets[va], out.probabilities)
                pipeline_s = fit_s + trans_s + fusion_s + graph_s + out.train_seconds + out.predict_seconds

                raw_rows.append({
                    "family": family,
                    "features": representation,
                    "fontes": "|".join(bundle),
                    "n_fontes": len(bundle),
                    "input_dim": xtr.shape[1],
                    "modelo": model_id,
                    "fold": fold,
                    "seed": seed,
                    "model_complexity": out.complexity,
                    "complexity_kind": out.complexity_type,
                    "feature_fit_seconds_incremental": fit_s,
                    "feature_transform_seconds_incremental": trans_s,
                    "feature_fusion_seconds": fusion_s,
                    "graph_build_seconds": graph_s,
                    "train_seconds": out.train_seconds,
                    "predict_seconds": out.predict_seconds,
                    "model_seconds": out.train_seconds + out.predict_seconds,
                    "pipeline_seconds_incremental": pipeline_s,
                    "condition_wall_seconds_observed": time.perf_counter() - t_cond,
                    **scores,
                })
                cm = class_metrics(dataset.targets[va], out.probabilities, dataset.classes)
                cm.insert(0, "seed", seed)
                cm.insert(0, "fold", fold)
                cm.insert(0, "modelo", model_id)
                cm.insert(0, "features", representation)
                cm.insert(0, "family", family)
                class_rows.append(cm)

experiment_result = standardize_experiment_result(
    pd.DataFrame(raw_rows),
    pd.concat(class_rows, ignore_index=True),
    plan,
)
print(f"Execução concluída: {len(experiment_result.conditions)} condições avaliadas!")

## 5 — Diagnósticos Relacionais, Fronteira de Pareto e `StudyResult`

A partir do `ExperimentResult` padronizado, a camada `veritas_ml.evaluation` calcula:
- **Ranking completo** das condições (`condition_summary`)
- **Fronteira de Pareto** (desempenho × tempo de pipeline)
- **Diagnóstico de Homofilia × Ganho Relacional** (`run_relational_diagnostics`)
- **Contrato declarativo `StudyResult`** pronto para `render_study(study_result)`

In [ ]:
from veritas_ml.evaluation import (
    assemble_study_result,
     run_relational_diagnostics,
    validate_study_result,
)

diagnostics = run_relational_diagnostics(dataset, feature_context, experiment_result, plan)
study_result = assemble_study_result(dataset, experiment_result, diagnostics=diagnostics)
assert validate_study_result(study_result)

print("=== Condição Vencedora (Referência) ===")
print(study_result.reference)

print("\n=== Ranking Consolidado (Reference Ranking) ===")
cols_ranking = [
    "representation_name",
    "model_name",
    "f1_micro_mean",
    "f1_macro_mean",
    "reference_score",
    "pipeline_incremental_mean",
]
display(study_result.tables["reference_ranking"][cols_ranking])

print("\n=== Fronteira de Pareto (Micro-F1 × Tempo) ===")
cols_pareto = ["representation_name", "model_name", "f1_micro_mean", "f1_macro_mean", "pipeline_incremental_mean"]
display(study_result.tables["pareto_primary"][cols_pareto])

print("\n=== Diagnóstico de Homofilia × Ganho do Grafo ===")
display(study_result.tables["diagnostic__relational_vs_individual"])